# TSsNorm — basic usage

**Task-Specific Sigmoid Normalization** turns any model that uses BatchNorm / LayerNorm
into a multi-task model, without adding feature extractors. Every normalization layer is
made *sigmoid-modulated* — its affine scale becomes `sigmoid(gamma)`, a bounded per-feature
scaler in `(0, 1)` — and *task-specific* — one copy per task, sharing everything else.

This notebook covers the whole surface except training: wrapping a model, feeding it to an
optimiser with discriminative learning rates, initialisation, task dispatch, reading out the
task-filter importance matrix, and converting a pretrained backbone. Sections follow the
layout of `tssnorm/utils.py`.

## A model to wrap

Any model works, as long as it normalizes with `nn.BatchNorm1d/2d` or `nn.LayerNorm`. (A
subclass of those is refused rather than silently left shared — see `register_norm`.) Here is
a tiny shared encoder with one head per task.

In [ ]:
import torch
from torch import nn

from tssnorm import wrap, sigmoidify, taskify, init_norms
from tssnorm import active_task, set_active_task
from tssnorm import get_param_group, get_other_param_group, SIG_BNS, SIG_LNS, SIG_NORMS
from tssnorm import get_importance

torch.manual_seed(0)
TASKS = ["seg", "depth", "normals"]


class MTL(nn.Module):
    def __init__(self, tasks):
        super().__init__()
        self.tasks = tasks
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8), nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(),
        )
        self.heads = nn.ModuleDict({t: nn.Conv2d(16, 1, 1) for t in tasks})

    def forward(self, x):
        out = {}
        for task in self.tasks:
            with active_task(self.encoder, task):
                out[task] = self.heads[task](self.encoder(x))
        return out


model = MTL(TASKS)
model.encoder

## Surgery: `wrap`

`wrap` is the whole method in one call. It runs three steps you can also invoke separately:

- `sigmoidify(m)` — replace each `BatchNorm`/`LayerNorm` with its sigmoid-modulated variant
- `taskify(m, tasks)` — replace each norm with a `TSNorm` holding one copy per task
- `init_norms(m, how)` — initialise the scalers (runs last, so a random init draws per task)

Each shared `BatchNorm2d` below becomes a `TSNorm` of three `SigBatchNorm2d`. Note that only
the norms are duplicated — the convolutions stay shared.

In [ ]:
wrap(model.encoder, TASKS)
model.encoder

In [ ]:
# the three verbs are equivalent to wrap(), for when norms need per-submodule treatment
fresh = MTL(TASKS).encoder
sigmoidify(fresh)
taskify(fresh, TASKS)
init_norms(fresh, how="const")
print("same structure as wrap():", repr(fresh) == repr(model.encoder))

## Initialisation

`init_norms(model, how=...)` sets the scalers `gamma`. The importance a task assigns to a
filter is `sigmoid(gamma)`:

- `const` (default) starts every scaler at the centre of its range, `sigmoid(0) = 0.5`, so no
  feature is preferred and each task can move freely in either direction
- `normal` / `uniform` spread the scalers across `(0, 1)`, so tasks start differentiated

The from-scratch experiments use `const`. A **pretrained** backbone instead inherits its
scalers through `convert` (see the last section, and the integrations for Tables 2–3).

In [ ]:
init_norms(model.encoder, how="const")
print("const :  importances all 0.5 ->", bool((get_importance(model.encoder) == 0.5).all()))

init_norms(model.encoder, how="normal")
I = get_importance(model.encoder)
print(f"normal:  importances in ({I.min():.2f}, {I.max():.2f}), std {I.std():.2f}")

init_norms(model.encoder, how="const");  # back to the paper default

## Task dispatch

A `TSNorm` normalises with whichever task is active. Set it for a whole forward pass with the
`active_task` context manager (used inside `MTL.forward` above) or `set_active_task`; the
context manager restores the previous task on exit. Forgetting to set one is a clear error,
not a silent wrong-task result.

In [ ]:
# give the tasks distinct scalers, so switching is observable without training
init_norms(model.encoder, how="normal")

x = torch.randn(2, 3, 16, 16)
model.eval()
with torch.no_grad():
    with active_task(model.encoder, "seg"):
        seg = model.encoder(x)
    with active_task(model.encoder, "depth"):
        depth = model.encoder(x)
print("seg and depth normalise differently:", not torch.allclose(seg, depth))

init_norms(model.encoder, how="const");  # restore the paper default

In [ ]:
# a norm that has never been pointed at a task raises, rather than guessing
never_set = MTL(TASKS).encoder
wrap(never_set, TASKS)
try:
    never_set(x)
except RuntimeError as e:
    print("no task set ->", str(e).split(";")[0])

## Optimiser: discriminative learning rates

The scalers are trained with a much higher learning rate than the weights they modulate
(`alpha = 100` for BN in the paper), so they allocate capacity before the backbone moves far.
Group the parameters **by layer type** and set a per-group `lr`:

- `get_param_group(model, of)` — parameters of every submodule of the given type(s)
- `get_other_param_group(model, excluding)` — everything else, split by parameter identity

BatchNorm and LayerNorm scalers get their own multipliers (`alpha = 100` and `10`), and the
rest excludes both. The three groups partition the model's parameters exactly; a group is
simply empty when the backbone has no norms of that kind — as with this BN-only encoder.

In [ ]:
lr = 1e-2
bn_scalers = get_param_group(model, SIG_BNS)
ln_scalers = get_param_group(model, SIG_LNS)                      # empty here: no LayerNorms
rest = get_other_param_group(model, excluding=SIG_NORMS)

counts = [sum(p.numel() for p in g["params"]) for g in (bn_scalers, ln_scalers, rest)]
print(f"BN scalers {counts[0]} + LN scalers {counts[1]} + rest {counts[2]} "
      f"== total {sum(p.numel() for p in model.parameters())}")

optim = torch.optim.SGD([
    {**bn_scalers, "lr": lr * 100},   # alpha_sBN
    {**ln_scalers, "lr": lr * 10},    # alpha_sLN
    {**rest, "lr": lr},
], momentum=0.9)
optim

## Importance readout

The task-filter importance matrix `I[t, i] = sigmoid(gamma[t, i])` is read straight off the
trained scalers, at no cost beyond training. It is the basis for the paper's capacity and
task-relationship analysis. Over the whole model it is `[tasks, filters]`; `per_layer=True`
gives one `[tasks, features]` matrix per layer (used for the specialisation-by-depth plots).

In [ ]:
I = get_importance(model.encoder)
print("importance matrix:", tuple(I.shape), "= [tasks, filters]")

per_layer = get_importance(model.encoder, per_layer=True)
print("per layer        :", [tuple(m.shape) for m in per_layer])

## Pretrained backbones

Modulating a *pretrained* norm means representing its learned scale with a sigmoid. Pass a
`convert` strategy so the new scalers reproduce the pretrained ones:

- `"clip"` — `gamma <- logit(clip(gamma))`, so `sigmoid(gamma)` matches the old scale
- `"clip+bias"` — also copies the bias and freezes it, avoiding a conversion shock

Use `init=None` with `wrap` so the initialiser does not overwrite the converted scalers.

In [ ]:
pretrained = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8), nn.ReLU(),
)
nn.init.uniform_(pretrained[1].weight, 0.2, 0.8)  # stand in for a trained scale

before = pretrained[1].weight.data.clone()
wrap(pretrained, TASKS, convert="clip+bias", init=None)

recovered = get_importance(pretrained)[0]  # sigmoid(gamma) for the first task
print("sigmoid(gamma) recovers the pretrained scale:",
      torch.allclose(recovered, before, atol=1e-6))